# Project 3: Linear Regression from Scratch

**Chapter:** 4 — Linear Regression

**Goal:** Implement OLS and ridge regression, compare the closed-form solution with gradient descent, and visualize the bias-variance tradeoff.

**What you'll build:**
1. Mean squared error loss and its gradient
2. Closed-form solution (normal equation)
3. Gradient descent solution
4. Ridge regression with varying λ

**Rules:** No sklearn. NumPy + matplotlib only.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['font.size'] = 12

## 1. Generate Data

In [ ]:
def make_regression_data(n=100, n_features=1, noise=0.5, seed=42):
    rng = np.random.default_rng(seed)
    X = rng.uniform(-3, 3, size=(n, n_features))
    true_w = rng.normal(size=n_features) * 2
    true_b = 1.5
    y = X @ true_w + true_b + rng.normal(scale=noise, size=n)
    return X, y, true_w, true_b

def make_polynomial_data(n=80, degree_true=3, noise=0.3, seed=42):
    """1D data from a polynomial — for testing overfitting."""
    rng = np.random.default_rng(seed)
    X = rng.uniform(-2, 2, size=(n, 1))
    y = 0.5 * X[:,0]**3 - X[:,0] + rng.normal(scale=noise, size=n)
    return X, y

# Simple 1D data
X_train, y_train, true_w, true_b = make_regression_data(n=100, n_features=1)

plt.scatter(X_train, y_train, alpha=0.6, s=30)
plt.xlabel('x'); plt.ylabel('y'); plt.title('Regression Data')
plt.show()
print(f"True relationship: y = {true_w[0]:.2f}·x + {true_b:.2f} + noise")

## 2. Linear Regression — Your Turn

### The math

**MSE loss:**
$$J(\mathbf{w}, b) = \frac{1}{n}\sum_{i=1}^{n}(\mathbf{w}\cdot\mathbf{x}_i + b - y_i)^2$$

**Closed-form (normal equation):**
$$\hat{\mathbf{w}} = (X^T X)^{-1} X^T \mathbf{y}$$
(where $X$ has a column of ones prepended for the bias)

**Gradient descent:**
$$\frac{\partial J}{\partial \mathbf{w}} = \frac{2}{n} X^T(X\mathbf{w} - \mathbf{y})$$

**Ridge (L2 regularization):**
$$\hat{\mathbf{w}} = (X^T X + \lambda I)^{-1} X^T \mathbf{y}$$

In [ ]:
class LinearRegression:
    def __init__(self):
        self.w = None  # includes bias as last element
    
    def _add_bias(self, X):
        """Prepend a column of ones to X. (Done for you.)"""
        return np.hstack([np.ones((X.shape[0], 1)), X])
    
    def fit_closed_form(self, X, y):
        """Fit using the normal equation: w = (X'X)^{-1} X'y
        
        X is augmented with a bias column.
        """
        ### YOUR CODE ###
        raise NotImplementedError("Normal equation")
    
    def fit_ridge(self, X, y, reg_lambda=1.0):
        """Fit ridge regression: w = (X'X + λI)^{-1} X'y
        
        Note: don't regularize the bias term (first element).
        """
        ### YOUR CODE ###
        raise NotImplementedError("Ridge regression closed form")
    
    def fit_gradient_descent(self, X, y, lr=0.01, epochs=1000):
        """Fit using gradient descent on MSE.
        
        Returns:
            list of MSE values per epoch
        """
        ### YOUR CODE ###
        raise NotImplementedError("GD on MSE")
    
    def predict(self, X):
        """Predict y = Xw (X is augmented with bias column)."""
        ### YOUR CODE ###
        raise NotImplementedError("X_aug @ self.w")
    
    def mse(self, X, y):
        preds = self.predict(X)
        return np.mean((preds - y) ** 2)

### Sanity checks

In [ ]:
reg = LinearRegression()
reg.fit_closed_form(X_train, y_train)
mse_cf = reg.mse(X_train, y_train)
print(f"Closed-form MSE: {mse_cf:.4f}")
print(f"Learned: bias={reg.w[0]:.3f}, slope={reg.w[1]:.3f}")
print(f"True:    bias={true_b:.3f}, slope={true_w[0]:.3f}")
assert mse_cf < 0.5, f"MSE too high: {mse_cf}"
print("✓ Closed-form looks correct")

reg_gd = LinearRegression()
losses = reg_gd.fit_gradient_descent(X_train, y_train, lr=0.01, epochs=500)
mse_gd = reg_gd.mse(X_train, y_train)
assert abs(mse_gd - mse_cf) < 0.1, f"GD and closed-form should be close: {mse_gd:.4f} vs {mse_cf:.4f}"
print(f"✓ GD MSE: {mse_gd:.4f} (close to closed-form)")

print("\n🎉 All checks passed!")

## 3. Visualize

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: fit line
ax1.scatter(X_train, y_train, alpha=0.5, s=30)
x_line = np.linspace(-3, 3, 100).reshape(-1, 1)
ax1.plot(x_line, reg.predict(x_line), 'r-', lw=2, label='Closed-form')
ax1.plot(x_line, reg_gd.predict(x_line), 'g--', lw=2, label='GD')
ax1.set_xlabel('x'); ax1.set_ylabel('y'); ax1.legend(); ax1.set_title('Linear Fit')

# Right: GD loss curve
ax2.plot(losses)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('MSE'); ax2.set_title('GD Loss Curve')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Experiment: Ridge Regularization

Use the polynomial data, create polynomial features, and see how ridge prevents overfitting.

In [ ]:
def polynomial_features(X, degree):
    """Create polynomial features [x, x², x³, ...] from 1D input."""
    return np.hstack([X ** d for d in range(1, degree + 1)])

X_poly, y_poly = make_polynomial_data(n=30, noise=0.5)
X_poly_test, y_poly_test = make_polynomial_data(n=100, noise=0.5, seed=99)

degree = 10
X_feat = polynomial_features(X_poly, degree)
X_feat_test = polynomial_features(X_poly_test, degree)

fig, axes = plt.subplots(1, 4, figsize=(24, 5))
x_plot = np.linspace(-2, 2, 200).reshape(-1, 1)
x_plot_feat = polynomial_features(x_plot, degree)

for lam, ax in zip([0, 0.01, 0.1, 10.0], axes):
    m = LinearRegression()
    if lam == 0:
        m.fit_closed_form(X_feat, y_poly)
    else:
        m.fit_ridge(X_feat, y_poly, reg_lambda=lam)
    
    y_plot = m.predict(x_plot_feat)
    train_mse = m.mse(X_feat, y_poly)
    test_mse = m.mse(X_feat_test, y_poly_test)
    
    ax.scatter(X_poly, y_poly, alpha=0.6, s=30)
    ax.plot(x_plot, y_plot, 'r-', lw=2)
    ax.set_ylim(-5, 5)
    ax.set_title(f"λ={lam}\ntrain={train_mse:.2f}, test={test_mse:.2f}")

plt.suptitle(f"Degree-{degree} Polynomial + Ridge", fontsize=14)
plt.tight_layout()
plt.show()

**Your observation:** *(Which λ gives the best test MSE? What happens at λ=0 and λ=10?)*

...

### A2: Regularization path
Plot how the regression coefficients change as λ varies. With degree-10 polynomial features, some coefficients explode without regularization — ridge tames them.

In [ ]:
lambdas_path = np.logspace(-3, 3, 30)
coef_paths = []

for lam in lambdas_path:
    m = LinearRegression()
    m.fit_ridge(X_feat, y_poly, reg_lambda=lam)
    coef_paths.append(m.w.copy())

coef_paths = np.array(coef_paths)  # (30, 11) — bias + 10 polynomial coefficients

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for j in range(coef_paths.shape[1]):
    label = 'bias' if j == 0 else f'$x^{{{j}}}$'
    ax1.semilogx(lambdas_path, coef_paths[:, j], '-', label=label, alpha=0.8)
ax1.set_xlabel('λ (log scale)'); ax1.set_ylabel('Coefficient value')
ax1.set_title('Regularization Path: Coefficients vs λ')
ax1.legend(fontsize=8, ncol=2); ax1.grid(True, alpha=0.3)

norms = np.linalg.norm(coef_paths[:, 1:], axis=1)  # exclude bias
ax2.semilogx(lambdas_path, norms, 'ko-', markersize=4)
ax2.set_xlabel('λ (log scale)'); ax2.set_ylabel('$\|w\|$ (excl. bias)')
ax2.set_title('Weight Norm Shrinkage')
ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## 5. Experiment: K-Fold Cross-Validation

A single train/test split can be noisy. K-fold CV gives a more robust estimate of generalization error.

### Your task
Implement `k_fold_cv(X, y, k=5, reg_lambda=0.1)` that:
1. Splits data into k roughly equal folds
2. For each fold: train ridge regression on the other k-1 folds, evaluate MSE on the held-out fold
3. Returns the array of k MSE values

In [ ]:
def k_fold_cv(X, y, k=5, reg_lambda=0.1):
    """K-fold cross-validation for ridge regression.
    
    Args:
        X: feature matrix (already polynomial-expanded if needed)
        y: targets
        k: number of folds
        reg_lambda: ridge regularization strength
    Returns:
        mse_scores: array of k MSE values, one per fold
    """
    ### YOUR CODE ###
    # 1. Compute fold size: n // k
    # 2. For each fold i:
    #    - X_val, y_val = fold i
    #    - X_tr, y_tr = everything else (np.concatenate the other folds)
    #    - Fit ridge regression on (X_tr, y_tr)
    #    - Evaluate MSE on (X_val, y_val)
    # 3. Return array of k MSE values
    raise NotImplementedError("K-fold cross-validation")


# --- Sanity checks ---
X_cv = polynomial_features(X_poly, degree=5)
cv_scores = k_fold_cv(X_cv, y_poly, k=5, reg_lambda=0.1)
assert len(cv_scores) == 5, f"Expected 5 scores, got {len(cv_scores)}"
assert all(s >= 0 for s in cv_scores), "MSE must be non-negative"
assert np.mean(cv_scores) < 5.0, f"Mean CV MSE too high: {np.mean(cv_scores):.2f}"
print(f"✓ 5-fold CV scores: {cv_scores.round(3)}")
print(f"  Mean: {np.mean(cv_scores):.3f} ± {np.std(cv_scores):.3f}")

# --- Lambda selection via CV ---
lambdas_cv = [0.001, 0.01, 0.1, 1.0, 10.0]
mean_scores = []
std_scores = []
for lam in lambdas_cv:
    scores = k_fold_cv(X_cv, y_poly, k=5, reg_lambda=lam)
    mean_scores.append(np.mean(scores))
    std_scores.append(np.std(scores))

fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(lambdas_cv, mean_scores, yerr=std_scores, fmt='bo-', capsize=5, markersize=8)
ax.set_xscale('log')
ax.set_xlabel('λ (log scale)'); ax.set_ylabel('Mean CV MSE')
ax.set_title('Cross-Validation: Selecting λ')
ax.grid(True, alpha=0.3)
best_lam = lambdas_cv[np.argmin(mean_scores)]
ax.axvline(best_lam, color='red', linestyle='--', alpha=0.5, label=f'Best λ={best_lam}')
ax.legend()
plt.tight_layout(); plt.show()
print(f"Best λ by CV: {best_lam}")

**Your observation:** *(Which λ minimizes CV error? How does this compare with the visual overfitting you saw above?)*

...

## 6. Experiment: Learning Curves

The most diagnostic plot in ML: train/test error vs dataset size. It tells you whether you need more data, more capacity, or more regularization.

In [ ]:
X_lc_pool, y_lc_pool = make_polynomial_data(n=200, noise=0.5, seed=42)
degree_lc = 5
X_lc_pool_feat = polynomial_features(X_lc_pool, degree_lc)
X_lc_test_feat = polynomial_features(X_poly_test, degree_lc)

sizes = [10, 20, 40, 80, 120, 160, 200]
train_mses, test_mses = [], []

for n in sizes:
    X_sub = X_lc_pool_feat[:n]
    y_sub = y_lc_pool[:n]
    m = LinearRegression()
    m.fit_ridge(X_sub, y_sub, reg_lambda=0.1)
    train_mses.append(m.mse(X_sub, y_sub))
    test_mses.append(m.mse(X_lc_test_feat, y_poly_test))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sizes, train_mses, 'b-o', label='Train MSE', markersize=7)
ax.plot(sizes, test_mses, 'r-s', label='Test MSE', markersize=7)
ax.set_xlabel('Training set size'); ax.set_ylabel('MSE')
ax.set_title('Learning Curves (degree=5, λ=0.1)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 7. What You Learned

1. When would you use the closed-form vs gradient descent?
2. What's the bias-variance tradeoff and how does λ control it?
3. Why don't we regularize the bias term?
4. What do learning curves tell you that a single train/test split doesn't?
5. Why is cross-validation more reliable than a single validation split?

---
**Next project:** Nonlinear Classification with Kernels — make linear methods handle curved boundaries.